# Lab 02 — Real Hubble Diagram: Residuals, Bootstrap and Statistical Evidence

**Course:** MSc Astrostatistics for Observational Cosmology  
**Format:** 4-hour hands-on practical + graded extension  
**Status:** **Graded Lab 1 (5%)**  
**Real data:** Pantheon+ / SH0ES

> ## Scientific question
> **What statistical structure is visible in the real Hubble diagram before a full cosmological fit?**

This lab develops the statistical habits needed for the rest of the course:

\[
\text{estimate} + \text{uncertainty} + \text{diagnostic} + \text{interpretation}.
\]

We use the published `MU_SH0ES` only as a real, calibrated distance-modulus product for exploratory Hubble-diagram statistics. It is **not** an independent measurement of \(H_0\), because its absolute scale already incorporates SH0ES calibration information.

The diagonal uncertainty column is used for visualization only. We do **not** fit cosmological parameters with it.

# 0. Learning outcomes

By the end you should be able to:

- construct a reference Hubble residual;
- explain the difference between a reference model and an inferred model;
- bootstrap at the correct physical unit;
- report an effect size with an interval rather than only a \(p\)-value;
- perform a permutation test without assuming a Gaussian sampling distribution;
- use robust summaries and empirical CDFs;
- distinguish descriptive structure from cosmological inference.

## 1. Five-line statistical statement

Complete before looking at the residual results.

**Estimand:**  
**Observed data:**  
**Generative assumptions:**  
**Uncertainty / selection:**  
**Validation / falsification test:**

In [ ]:
from pathlib import Path
import io, sys, warnings, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from astropy.cosmology import FlatLambdaCDM

SEED = 2026
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", 60)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 2. Load the same real Pantheon+ table

In [ ]:
PANTHEON_URL = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/"
    "Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES.dat"
)

def load_pantheon(url=PANTHEON_URL):
    """Download the published Pantheon+SH0ES high-level table."""
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), sep=r"\s+")

sn = load_pantheon()
print(f"Rows: {len(sn):,}")
print(f"Unique CID values: {sn['CID'].nunique():,}")
display(sn.head())

## 3. Build an object-level exploratory table

Pantheon+ can contain multiple light-curve entries for one physical SN. For this **descriptive resampling lab**, we collapse repeated `CID` rows so that our bootstrap unit is the physical SN.

This object-level table is an instructional construct for descriptive statistics only; it is **not** a replacement for the released covariance-aware cosmology likelihood.

We use medians across repeated rows to avoid treating one SN observed by multiple surveys as several independent bootstrap objects.

In [ ]:
cols = ["CID","zHD","MU_SH0ES","MU_SH0ES_ERR_DIAG","IS_CALIBRATOR"]

obj = (
    sn[cols]
    .replace([np.inf,-np.inf], np.nan)
    .dropna(subset=["CID","zHD","MU_SH0ES"])
    .groupby("CID", as_index=False)
    .agg(
        zHD=("zHD","median"),
        MU_SH0ES=("MU_SH0ES","median"),
        MU_ERR_DIAG=("MU_SH0ES_ERR_DIAG","median"),
        IS_CALIBRATOR=("IS_CALIBRATOR","max")
    )
)

# We exclude very local objects for this descriptive Hubble-flow exercise.
analysis = obj[(obj["zHD"] > 0.01) & (obj["IS_CALIBRATOR"] == 0)].copy()

print("Unique physical SNe in object-level table:", len(obj))
print("Analysis objects after descriptive cuts:", len(analysis))
display(analysis.head())

### Question

Why do we exclude Cepheid calibrators from this particular Hubble-flow residual exercise?

Why does this **not** mean calibrators are unimportant for the distance ladder?

# Part I — Reference Hubble diagram

## 4. Define a reference cosmology

We choose a reference flat-\(\Lambda\)CDM curve:

\[
H_0^{\rm ref}=73~{\rm km\,s^{-1}\,Mpc^{-1}},
\qquad
\Omega_m^{\rm ref}=0.30.
\]

These values are **not estimated in this lab**.

They define a baseline against which residual structure is displayed.

In [ ]:
H0_REF = 73.0
OM0_REF = 0.30

cosmo_ref = FlatLambdaCDM(H0=H0_REF, Om0=OM0_REF, Tcmb0=2.725)

analysis["MU_REF"] = cosmo_ref.distmod(analysis["zHD"].to_numpy()).value
analysis["RESID"] = analysis["MU_SH0ES"] - analysis["MU_REF"]

display(analysis[["CID","zHD","MU_SH0ES","MU_REF","RESID"]].head())

## 5. Plot the Hubble diagram and residuals

A useful Hubble-diagram figure should separate the large monotonic distance–redshift trend from the much smaller residual structure.

In [ ]:
zgrid = np.geomspace(0.01, max(analysis["zHD"])*1.03, 500)
mugrid = cosmo_ref.distmod(zgrid).value

fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(analysis["zHD"], analysis["MU_SH0ES"], s=9, alpha=0.55)
ax.plot(zgrid, mugrid, lw=2, label="Reference flat ΛCDM")
ax.set_xscale("log")
ax.set_xlabel(r"$z_{\rm HD}$")
ax.set_ylabel(r"$\mu$ [mag]")
ax.set_title("Pantheon+ / SH0ES Hubble diagram")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8,4))
ax.axhline(0, lw=1)
ax.scatter(analysis["zHD"], analysis["RESID"], s=9, alpha=0.55)
ax.set_xscale("log")
ax.set_xlabel(r"$z_{\rm HD}$")
ax.set_ylabel(r"$\Delta\mu$ [mag]")
ax.set_title("Residuals relative to the reference cosmology")
plt.show()

### Exercise 1

Calculate:

- mean residual;
- median residual;
- standard deviation;
- MAD;
- 16th, 50th and 84th percentiles.

Then answer:

> If the mean and median differ, what does that tell you—and what does it **not** tell you?

In [ ]:
r = analysis["RESID"].to_numpy()

# TODO: fill these values.
summary = {
    "mean": np.nan,
    "median": np.nan,
    "std": np.nan,
    "MAD": np.nan,
    "p16": np.nan,
    "p50": np.nan,
    "p84": np.nan
}
summary

# Part II — Bootstrap uncertainty

## 6. Why bootstrap physical objects?

The target statistic is the median residual of the population represented by this sample.

We resample **rows of `analysis`**, because each row now corresponds to one physical `CID`.

For bootstrap sample \(b\):

\[
\tilde r^{(b)}
=
{\rm median}
\left(
r_{i_1},r_{i_2},\ldots,r_{i_N}
\right).
\]

Repeat many times to approximate the sampling distribution of the median.

In [ ]:
def bootstrap_statistic(x, statistic=np.median, n_boot=5000, rng=None):
    """Bootstrap a 1-D statistic by resampling observations with replacement."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    rng = np.random.default_rng() if rng is None else rng
    out = np.empty(n_boot)
    n = len(x)
    for b in range(n_boot):
        sample = rng.choice(x, size=n, replace=True)
        out[b] = statistic(sample)
    return out

# TODO: call the function on analysis["RESID"].
boot_global = None

### Exercise 2

Using at least 5,000 bootstrap samples:

1. estimate the median residual;
2. report the 2.5th and 97.5th percentiles of the bootstrap distribution;
3. plot the bootstrap distribution;
4. mark the observed median.

**Do not** call this interval a cosmological-parameter confidence interval. It is an interval for a descriptive residual statistic under this resampling procedure.

In [ ]:
# TODO: after computing boot_global
# ci95 = np.percentile(boot_global, [2.5,97.5])
# ...

# Part III — A pre-specified redshift comparison

## 7. Freeze the comparison before testing

Set

\[
z_{\rm split}=0.30.
\]

Define

- low-\(z\): \(z<0.30\);
- high-\(z\): \(z\ge0.30\).

We will compare the median residuals.

Our effect size is

\[
\Delta_{\rm med}
=
{\rm median}(r_{\rm high})
-
{\rm median}(r_{\rm low}).
\]

The sign and magnitude of \(\Delta_{\rm med}\) are scientifically more informative than a \(p\)-value alone.

In [ ]:
Z_SPLIT = 0.30

low = analysis.loc[analysis["zHD"] < Z_SPLIT, "RESID"].to_numpy()
high = analysis.loc[analysis["zHD"] >= Z_SPLIT, "RESID"].to_numpy()

print("N low-z:", len(low))
print("N high-z:", len(high))

# TODO:
# effect_obs = np.median(high) - np.median(low)

## 8. Bootstrap the difference in medians

Resample the low- and high-\(z\) groups **within group**, then recompute the difference in medians.

Report:

\[
\Delta_{\rm med}
\]

and a 95% bootstrap interval.

In [ ]:
def bootstrap_difference_medians(a, b, n_boot=5000, rng=None):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    rng = np.random.default_rng() if rng is None else rng
    out = np.empty(n_boot)
    for k in range(n_boot):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        out[k] = np.median(bb) - np.median(aa)
    return out

# TODO:
# boot_diff = bootstrap_difference_medians(low, high, 5000, rng)
# np.percentile(boot_diff, [2.5,50,97.5])

# Part IV — Permutation test

## 9. Null hypothesis

For this exercise:

\[
H_0:
\]

the labels “low-\(z\)” and “high-\(z\)” are exchangeable with respect to the residual distribution.

Under \(H_0\), repeatedly shuffle the group labels while preserving group sizes.

Use the **absolute difference in medians** as the test statistic.

\[
T=
\left|
{\rm median}(r_{\rm high})
-
{\rm median}(r_{\rm low})
\right|.
\]

The permutation \(p\)-value is the fraction of shuffled datasets producing \(T\) at least as large as observed.

In [ ]:
def permutation_test_difference_medians(a, b, n_perm=10000, rng=None):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    rng = np.random.default_rng() if rng is None else rng

    observed = abs(np.median(b)-np.median(a))
    pooled = np.concatenate([a,b])
    n_a = len(a)

    null = np.empty(n_perm)
    for k in range(n_perm):
        perm = rng.permutation(pooled)
        aa = perm[:n_a]
        bb = perm[n_a:]
        null[k] = abs(np.median(bb)-np.median(aa))

    # plus-one correction
    p = (np.sum(null >= observed)+1)/(n_perm+1)
    return observed, p, null

# TODO:
# t_obs, p_perm, perm_null = permutation_test_difference_medians(low, high, 10000, rng)

### Exercise 3

Produce:

- observed effect size;
- 95% bootstrap interval for the signed effect;
- permutation \(p\)-value;
- histogram of the permutation null distribution with the observed \(T\) marked.

Then write 3–5 sentences interpreting all three together.

Avoid:

> “\(p<0.05\), therefore the cosmology is wrong.”

That conclusion is not supported by this exercise.

# Part V — Empirical CDF and robust interpretation

## 10. ECDF

An ECDF uses every observation without requiring histogram bins.

For ordered values \(x_{(i)}\),

\[
\widehat F(x_{(i)})=\frac{i}{N}.
\]

In [ ]:
def ecdf(x):
    x = np.sort(np.asarray(x, float))
    y = np.arange(1, len(x)+1)/len(x)
    return x, y

x1,y1 = ecdf(low)
x2,y2 = ecdf(high)

fig, ax = plt.subplots(figsize=(7,4))
ax.step(x1,y1,where="post",label=f"z < {Z_SPLIT}")
ax.step(x2,y2,where="post",label=f"z ≥ {Z_SPLIT}")
ax.set_xlabel(r"$\Delta\mu$ [mag]")
ax.set_ylabel("ECDF")
ax.set_title("Hubble-residual ECDFs")
ax.legend()
plt.show()

### Exercise 4

From the ECDF alone, comment on:

- location shift;
- spread;
- tails;
- whether one distribution appears systematically offset.

Do not over-interpret small visual separations.

# 11. In-class checkpoint

Before leaving the practical, show:

1. Hubble diagram;
2. residual-versus-redshift plot;
3. global bootstrap distribution;
4. \(\Delta_{\rm med}\) with bootstrap interval;
5. permutation null distribution and \(p\)-value;
6. low/high-\(z\) ECDF.

You should be able to explain **why each figure exists**.

# Extended Analysis — Graded Lab 1

Complete after class in the **same notebook**.

## E1. Robust versus classical summaries

Construct a table comparing for the full residual sample and the two redshift groups:

- mean;
- standard deviation;
- median;
- MAD.

Discuss which pair is more stable to tails/outliers in this dataset.

## E2. Sensitivity to the redshift boundary

Repeat only the **effect-size + bootstrap interval** calculation for:

\[
z_{\rm split}=0.25
\quad\text{and}\quad
z_{\rm split}=0.35.
\]

Do not choose a boundary because it gives the smallest \(p\)-value.

Create a table:

| split | N low | N high | \(\Delta_{\rm med}\) | 95% bootstrap interval |
|---|---:|---:|---:|---:|

## E3. Influence diagnostic

Identify the five SNe with largest absolute residuals.

Recalculate the global median and \(\Delta_{\rm med}\) after excluding those five objects.

This is a **sensitivity analysis**, not permission to remove inconvenient data from a scientific fit.

## E4. Scientific conclusion — 300–500 words

Answer all six:

1. What quantity did you estimate?
2. What is the main numerical effect size?
3. What uncertainty accompanies it?
4. What does the permutation test add?
5. How sensitive is the result to the redshift split and extreme residuals?
6. What cosmological claim would be unjustified from this lab?

### Required final sentence

Complete:

> “From this lab alone, we can conclude ________, but we cannot conclude ________.”

# 12. Submission checklist

Your submitted notebook must:

- run from top to bottom after kernel restart;
- retain the fixed random seed;
- contain all required plots;
- report effect sizes **and** uncertainty;
- include the sensitivity table;
- include the 300–500 word Scientific Conclusion;
- use no synthetic astronomical objects.

**Grading:** scientific formulation 20%, statistical correctness 25%, uncertainty 20%, diagnostics 15%, reproducibility 10%, interpretation 10%.

# 13. Bridge to Lab 03

Next week we move from:

\[
\text{“What structure is in the residuals?”}
\]

to

\[
\text{“Which cosmological parameter values maximize a likelihood?”}
\]

That is the transition from **descriptive/resampling statistics** to **parameter inference**.